In [2]:
import asyncio

async def hello():
    await asyncio.sleep(2)
    print("hello")

print("before")

await hello() # Do this in Notebook

print("after")

# asyncio.run(hello()) # Do this in .py file


before
hello
after


In [4]:
import asyncio

async def fetch_data():
    print("Fetching data...")

    await asyncio.sleep(2)

    print("Data received!")
    return "Some data"


async def main():
    data = await fetch_data()
    print(data)

await main()

Fetching data...
Data received!
Some data


In [5]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")


async def main():
    await task_a()
    await task_b()


start = time.perf_counter()

await main()

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task A finished
Task B started
Task B finished
Time taken: 3.02 seconds


In [6]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")

    return "Result A"


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")

    return "Result B"


async def main():
    results = await asyncio.gather(
        task_a(),
        task_b(),
    )

    print(results)


start = time.perf_counter()

await main()

# asyncio.run(main())

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task B started
Task B finished
Task A finished
['Result A', 'Result B']
Time taken: 2.01 seconds


In [7]:
import asyncio

async def generate_numbers():

    for number in range(5):
        await asyncio.sleep(2)
        yield number


async def main():

    async for number in generate_numbers():
        print(number)


await main()

0
1
2
3
4


In [10]:
import asyncio
import httpx


async def fetch_post():

    async with httpx.AsyncClient(verify=False) as client:

        response = await client.get(
            "https://jsonplaceholder.typicode.com/posts/1"
        )

        return response.json()


async def main():
    post = await fetch_post()
    print(post)


await main()

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [11]:
import asyncio
import httpx


async def fetch_user(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/users/1"
    )

    return response.json()


async def fetch_post(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/posts/1"
    )

    return response.json()


async def fetch_todo(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/todos/1"
    )

    return response.json()


async def main():

    async with httpx.AsyncClient(verify=False) as client:

        user, post, todo = await asyncio.gather(
            fetch_user(client),
            fetch_post(client),
            fetch_todo(client),
        )

    print("User:")
    print(user["name"])

    print("\nPost:")
    print(post["title"])

    print("\nTodo:")
    print(todo["title"])


await main()

User:
Leanne Graham

Post:
sunt aut facere repellat provident occaecati excepturi optio reprehenderit

Todo:
delectus aut autem


In [13]:
from pydantic import BaseModel, ValidationError

class ChatRequest(BaseModel):
    message: str
    temperature: float
    model: str
    dictionary: dict[str, int]

# Automatic type coercion when compatible (e.g., string "0.7" converted to float 0.7)
valid_req = ChatRequest(message="Explain vector databases", temperature="0.7", model="gpt-4o", dictionary={"key": 1})
print(f"Valid request object created: {valid_req}")
print(f"Temperature type is float: {type(valid_req.temperature)}")

# Automatic validation error on incompatible data types
try:
    ChatRequest(message=123, temperature="invalid_number", model="gpt-4o")
except ValidationError as e:
    print("\n--- Caught expected Pydantic ValidationError ---")
    print(e)

Valid request object created: message='Explain vector databases' temperature=0.7 model='gpt-4o' dictionary={'key': 1}
Temperature type is float: <class 'float'>

--- Caught expected Pydantic ValidationError ---
3 validation errors for ChatRequest
message
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='invalid_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing
dictionary
  Field required [type=missing, input_value={'message': 123, 'tempera...ber', 'model': 'gpt-4o'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [14]:
class ChatRequestOptional(BaseModel):
    message: str                       # Required field
    temperature: float | None = None   # Optional field, defaults to None
    max_tokens: int = 1000             # Optional field, defaults to 1000
    system_prompt: str | None          # Explicitly required field that accepts None!

# Passing required fields
req1 = ChatRequestOptional(message="Hello", system_prompt=None)
print(f"Req 1: {req1}")

# Failing because 'system_prompt' was omitted even though it accepts None
try:
    ChatRequestOptional(message="Hello")
except ValidationError as e:
    print("\n--- Caught error when missing required field 'system_prompt' ---")
    print(e)

Req 1: message='Hello' temperature=None max_tokens=1000 system_prompt=None

--- Caught error when missing required field 'system_prompt' ---
1 validation error for ChatRequestOptional
system_prompt
  Field required [type=missing, input_value={'message': 'Hello'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [16]:
class ModelConfig(BaseModel):
    model: str
    temperature: float

class Message(BaseModel):
    role: str
    content: str

class NestedChatRequest(BaseModel):
    config: ModelConfig
    messages: list[Message]

raw_payload = {
    "config": {"model": "gpt-4o", "temperature": 0.7},
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is RAG?"}
    ]
}

req = NestedChatRequest(**raw_payload)
print(req.config)
print(req.messages)

model='gpt-4o' temperature=0.7
[Message(role='system', content='You are a helpful assistant.'), Message(role='user', content='What is RAG?')]


In [17]:
from pydantic import Field

class ConstrainedChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=5000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    # Alias maps incoming key 'user_id' to internal variable name 'user_identifier'
    user_identifier: str = Field(alias="user_id")

# Passing payload using the alias name 'user_id'
req = ConstrainedChatRequest(message="Hello", user_id="usr_12345", temperature=1.4)
print(f"Parsed with Alias -> Internal Attribute: {req}")

Parsed with Alias -> Internal Attribute: message='Hello' temperature=1.4 user_identifier='usr_12345'


In [18]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

class ConstrainedChatRequest(BaseModel):
    message: Annotated[str, Field(min_length=1, max_length=5000)]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

In [20]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

# ------------------------------------------------------------------
# Reusable Custom Domain Types with Annotated
# ------------------------------------------------------------------
PromptText = Annotated[
    str, 
    Field(min_length=1, max_length=5000, description="The primary user prompt")
]
Temperature = Annotated[
    float, 
    Field(ge=0.0, le=2.0, description="Sampling temperature")
]

# ------------------------------------------------------------------
# Model Definition using Modern Annotated Syntax
# ------------------------------------------------------------------
class ConstrainedChatRequest(BaseModel):
    message: PromptText
    temperature: Temperature = 0.7
    
    # Inline Annotated field with alias mapping
    user_identifier: Annotated[str, Field(alias="user_id")]

req = ConstrainedChatRequest(message="Explain Vector DBs", user_id="usr_12345")
print(f"Parsed with Alias -> Internal Attribute: {req.user_identifier}")
print(f"Default Temperature Applied: {req.temperature}")

Parsed with Alias -> Internal Attribute: usr_12345
Default Temperature Applied: 0.7
